# Build 04-02 · shared-claim SHAP — v2 (log) & v3, restricted to the v2<->v3 overlap

Computes the SHAP attributions `04_02_shap_did_concentration.ipynb` §5/§6/§6b need for the
**paired shared-claim** comparison (`estimator/effect/shap_did.py`'s module docstring, section
"Paired shared-claim variant") — the one computation that notebook cannot do itself, because it
needs each version's OWN model pickle, which only opens inside that version's own env. This
notebook is that computation, kept separate for the same reason `00_SHAP.ipynb` is its own
notebook rather than a cell inside an analysis-env one.

**What "shared claim" means here.** `corrector_targets_v3_<split>.parquet`
(`03_01_corrector_inputs.ipynb`) already joins v3's own claims to v2's PRODUCTION-LOG score and
decision on the same `claim_id` — a claim v2 scored in production, that later became one of v3's
own train/oot rows. That join is done; what is missing is SHAP on both sides of it:

- **v2 side ("before")** — v2's OWN model, explained against `log_features` (v2's SERVING-time
  matrix — the only place these claims' v2-side feature values live; v2's existing `attributions`
  are from its 2018-2020 TRAIN/VAL/TEST fit, a completely different era and claim set).
- **v3 side ("after")** — v3's OWN model, explained against its normal `processed_inputs`, but
  restricted to the SAME claim_ids as the v2 side (v3's existing `attributions` are an
  unrestricted random sample and are not guaranteed to cover these specific claims).

Both sides use the identical claim_id set per split, so the two attribution files are a genuinely
PAIRED sample — not two independent samples that happen to overlap.

## How to run

**Run this notebook TWICE — once per kernel, either order:**
1. Select the `env-v2` Jupyter kernel, run every cell. Writes
   `v2_log_attributions_shared_v3_<split>.parquet` for every split in `V3_SPLITS` (§1).
2. Select the `env-v3` kernel, run every cell (same file — no edits needed). Writes
   `v3_attributions_shared_<split>.parquet`.

`VERSION` is auto-detected from the kernel (`shap_kit.detect_version()`, the same mechanism
`00_SHAP.ipynb` uses) — running under the wrong kernel fails loudly in §0 instead of silently
producing attributions for the wrong model. **v1 is out of scope**: it has no production log at
all (destroyed), so there is no v1<->v2 shared-claim analogue to build here — see
`project_v1v2_inheritance_dead` (v1->v2 has no log and is structurally inestimable).

Output lands in `src/data/real/estimation/shared_claim_shap/` —
`shared_claim_paths()`/`SHARED_CLAIM_DIR` in `estimator/effect/shap_did.py` read from exactly
this directory, so nothing downstream needs to change once both kernel runs are done.

In [ ]:
# §0 -- setup + which kernel is this? (mirrors 00_SHAP.ipynb's own VERSION-detection cell)
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd()
while not (ROOT / "src" / "config.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import config
import schema
import shap_kit as sk
from trained_order import select_features

sk.style()

# Inferred from the interpreter path (src/envs/v2/.venv/... or src/envs/v3/.venv/...) -- the same
# mechanism 00_SHAP.ipynb uses, so a mis-selected kernel fails HERE (env_report's xgboost check)
# rather than quietly writing attributions for the wrong model.
VERSION = sk.detect_version()
ENV = sk.env_report(VERSION, strict=True)
VERSION = ENV["version"]
assert VERSION in ("v2", "v3"), (
    f"this notebook is v2/v3 only -- got VERSION={VERSION!r}. v1 has no production log at all "
    f"(destroyed), so there is no v1<->v2 shared-claim pair to build (project_v1v2_inheritance_dead: "
    f"v1->v2 has no log and is structurally inestimable). Run this under the env-v2 or env-v3 kernel."
)
print(f"\nrunning as {VERSION}")


In [ ]:
# §1 -- RUN_SPEC: what to compute, how much, where to write it
ID_COL = schema.CLAIM_ID
SOURCE = "real"

# Always v3's OWN split names -- corrector_targets is a v3 artefact (v3's targets LEFT-joined to
# v2's log treatment) regardless of which side (v2 or v3) THIS kernel is explaining.
V3_SPLITS = list(dict.fromkeys(["train", config.OOT_SPLIT["v3"]]))

# attribute.py / this notebook has NO built-in downsampling once a claim_id list is handed over --
# it explains every id it is given. corrector_targets can be large (real v2<->v3 overlap counts
# reported 2026-09-22: train ~179,586, oot ~31,264 shared claims -- see this repo's own memory,
# feedback-dummy-data-marker, for why a number like that must come from the user/a real run, never
# be assumed from a local file). EXPLAIN_CAP matches 00_SHAP.ipynb's own N_EXPLAIN default (5000)
# -- SHAP cost is roughly linear in this, so it is the project's already-established "finishes in a
# sitting" budget, not a new number invented here. Override below if one split needs a different cap.
EXPLAIN_CAP = 5000
BACKGROUND_N = 500     # matches 00_SHAP.ipynb's N_BACKGROUND
SEED = 0
# EXPLICIT, never "auto": shared_claim_table()'s concentration.require_comparable() call RAISES
# if the v2-side and v3-side attributions end up on different backends -- comparability here is
# load-bearing (estimator/effect/shap_did.py's module docstring), so this must not silently pick
# whatever each kernel's env happens to default to.
BACKEND = "shap"

OUT_DIR = ROOT / "src" / "data" / "real" / "estimation" / "shared_claim_shap"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"v3 splits   : {V3_SPLITS}")
print(f"explain cap : {EXPLAIN_CAP:,} claims/split (seed={SEED})")
print(f"background  : {BACKGROUND_N}")
print(f"backend     : {BACKEND}")
print(f"out dir     : {OUT_DIR}")


In [ ]:
# §2 -- the model (VERSION's own production pickle -- config.path('model', VERSION, 'real'):
# v2's is DECLARED (outputs/model.pkl, the LIVE model), v3's is DECLARED too (outputs/p146_model.pkl)
# -- neither falls back to our own retrained baseline. Using anything else here would explain a
# DIFFERENT model's reasoning than the one that actually produced these claims' scores/decisions.
model_path = config.path("model", VERSION, SOURCE)
print(f"model: {model_path}")
est = sk.load_estimator(model_path)
print(f"  {type(est).__name__}")


In [ ]:
# §3 -- v2's log_features is ONE file shared by every split (v2's log is not split at all --
# it is filtered down to whichever claim_ids a given split's corrector_targets names). Load it
# ONCE here rather than once per split inside §4's loop. v3 has no equivalent: its processed_inputs
# is genuinely per-split, so that side reads fresh inside the loop instead.
if VERSION == "v2":
    log_features_path = config.path("log_features", "v2", SOURCE)
    if not log_features_path.is_file():
        raise SystemExit(
            f"{log_features_path} missing -- run notebook/real/01_export_v2_logs.ipynb first "
            f"(it writes log_raw/log_features/log_scores/log_targets/log together, so if "
            f"corrector_targets already exists, this file almost certainly does too -- check the "
            f"path above is right before assuming it is genuinely missing)."
        )
    log_features_df = pd.read_parquet(log_features_path)
    print(f"log_features: {log_features_path}  {log_features_df.shape}")
else:
    log_features_df = None   # v3 reads its own per-split processed_inputs inside §4 instead


In [ ]:
# §4 -- per v3 split: explain-ids from corrector_targets (capped, seed=0) -> select this
# VERSION's feature columns -> SHAP -> write attributions + meta into OUT_DIR.
#
# The SAME corrector_targets file is the explain-id source for BOTH kernel runs, and the cap
# below is a DETERMINISTIC function of that file's own content (same seed, same source frame) --
# so the v2 kernel run and the v3 kernel run land on the IDENTICAL sampled claim_id set with no
# shared state file needed between them. That identity is what makes the two output files a
# PAIRED sample rather than two independent ones that happen to overlap.
summary_rows = []

for v3_split in V3_SPLITS:
    print(f"\n=== v3/{v3_split} ({VERSION} side) ===")

    explain_path = config.split_path("corrector_targets", "v3", v3_split)
    if not explain_path.is_file():
        print(f"  {explain_path} missing -- run 03_01_corrector_inputs.ipynb first. skipping.")
        continue
    ct = pd.read_parquet(explain_path)
    ids_full = ct[[ID_COL]].drop_duplicates()
    if len(ids_full) > EXPLAIN_CAP:
        ids = (ids_full.sample(n=EXPLAIN_CAP, random_state=SEED)
                        .sort_values(ID_COL).reset_index(drop=True))
        print(f"  {len(ids_full):,} shared claims -> capped to {len(ids):,} (seed={SEED})")
    else:
        ids = ids_full
        print(f"  {len(ids):,} shared claims (under the {EXPLAIN_CAP:,} cap -- using all of them)")

    if VERSION == "v2":
        df, features_path = log_features_df, log_features_path
    else:
        features_path = config.split_path("processed_inputs", "v3", v3_split)
        if not features_path.is_file():
            print(f"  {features_path} missing -- skipping this split.")
            continue
        df = pd.read_parquet(features_path)
        print(f"  processed_inputs: {features_path}  {df.shape}")

    if ID_COL not in df.columns:
        raise SystemExit(f"{features_path} has no {ID_COL!r} column (columns: {list(df.columns)[:10]})")

    id_set = set(ids[ID_COL])
    sub = df[df[ID_COL].isin(id_set)].copy()
    n_missing = len(id_set) - sub[ID_COL].nunique()
    if n_missing:
        print(f"  {n_missing:,} of {len(id_set):,} requested ids are absent from "
              f"{features_path.name} -- continuing with the {sub[ID_COL].nunique():,} present")
    if sub.empty:
        print("  none of the requested ids are in this features file -- skipping this split.")
        continue

    n_dup = int(sub[ID_COL].duplicated().sum())
    if n_dup:
        raise SystemExit(
            f"{features_path.name} has {n_dup} duplicate {ID_COL} row(s) among the requested "
            f"ids. log_features carries one row per SCORING EVENT (claim_id + correlation_id), "
            f"not guaranteed one row per claim -- see this notebook's own markdown cell. Collapse "
            f"to one row per claim first: the same rule 03_01_corrector_inputs.ipynb's "
            f"collapse_events() uses (a decision=1 event wins, else the max-score event) -- needs "
            f"log_scores joined back in by correlation_id to know which event that was."
        )

    feature_cols, order_source = select_features(sub, est, ID_COL, registry_path=None)
    X = sub[feature_cols]
    # shap's TreeExplainer (shap/explainers/_tree.py, self.trees = ...) requires float64 --
    # a non-float64 column (int/category/etc. surviving from the parquet) raises there rather
    # than in this notebook. Same cast as 00_SHAP.ipynb §2 (added 2026-08-24 for the identical
    # v3 collapse); this notebook never had it, so v3's kernel run hit it fresh.
    non_float = X.dtypes[X.dtypes != "float64"]
    if len(non_float):
        print(f"  casting {len(non_float)} non-float64 column(s) to float64 for shap: "
              f"{list(non_float.index[:8])}" + (" …" if len(non_float) > 8 else ""))
        X = X.astype("float64")
    print(f"  X: {X.shape}  (feature order: {order_source})")

    bg = X.sample(n=min(BACKGROUND_N, len(sub)), random_state=SEED + 1)
    att = sk.compute(est, X, background=bg, backend=BACKEND)
    print(f"  {att!r}")
    # non-fatal by design (shap_kit.check_additivity prints a verdict and returns the gap,
    # never raises) -- shap's OWN internal additivity check is disabled inside
    # attribution/backend.py's _via_shap_backend() for exactly this reason: it raised and
    # discarded an already-computed phi over a ~3e-4 gap on a ~5 margin (2026-09-23), well
    # inside its own stated tolerance band -- almost certainly xgboost-vs-TreeSHAP floating
    # point noise, not a wrong X. A gap near the MARGIN's own size here would mean something
    # really is wrong (wrong column order, wrong estimator) -- read this print, don't ignore it.
    sk.check_additivity(att, est)

    if VERSION == "v2":
        stem = f"v2_log_attributions_shared_v3_{v3_split}"
        role, split_label = "before", f"log_shared_v3_{v3_split}"
    else:
        stem = f"v3_attributions_shared_{v3_split}"
        role, split_label = "after", v3_split
    out_path = OUT_DIR / f"{stem}.parquet"
    meta_path = out_path.with_name(out_path.stem + "_meta.json")

    out = att.frame(id_values=sub[ID_COL].values, id_col=ID_COL)
    out.to_parquet(out_path, index=False)

    meta = {
        "version": VERSION, "split": split_label, "role": role,
        "backend": att.backend, "perturbation": att.perturbation, "model_output": "raw",
        "model_path": str(model_path), "features_path": str(features_path),
        "explain_ids_file": str(explain_path),
        "n_shared_claims_total": int(len(ids_full)),
        "explain_cap": EXPLAIN_CAP,
        "n_rows": int(len(out)), "n_features": len(feature_cols),
        "feature_names": feature_cols, "feature_order": order_source,
        "background_n": int(len(bg)), "seed": SEED,
        "base_value": float(np.mean(att.base)),
    }
    meta_path.write_text(json.dumps(meta, indent=2), encoding="utf-8")
    print(f"  -> {out_path.relative_to(ROOT)}")

    mabs = att.mean_abs
    top_str = ", ".join(f"{n}={v:.4f}" for n, v in mabs.head(5).items())
    print(f"  top mean|phi|: {top_str}")

    summary_rows.append({"v3_split": v3_split, "role": role, "n_shared_total": len(ids_full),
                          "n_explained": len(out), "n_features": len(feature_cols),
                          "out_path": str(out_path.relative_to(ROOT))})

print()
if summary_rows:
    display(pd.DataFrame(summary_rows).set_index("v3_split"))
else:
    print("nothing written this run -- see the per-split messages above.")


## Notes

- **Why this is a separate notebook, not a cell in `04_02_shap_did_concentration.ipynb`.** SHAP
  needs the model FUNCTION, not just its scores, so it must open the pickle -- and a pickle only
  unpickles inside the env it was serialised in. `04_02_shap_did_concentration.ipynb` runs in the
  shared analysis `.venv` and never opens a model, same as every other notebook that reads
  `attributions` — this one is Version Layer work (`attribution/attribute.py`'s own module
  docstring makes the identical argument for why IT is a separate process).
- **Why not just call `attribute.py`.** `attribute.py`'s CLI already does exactly this
  computation and could have been shelled out to per split/version instead — this notebook exists
  so the same run is inspectable interactively (kernel-per-version, like `00_SHAP.ipynb`) rather
  than a batch of CLI invocations with hand-built `--out` paths. It calls the SAME underlying
  functions `attribute.py` does (`shap_kit.compute()` -> `attribution/backend.py`'s
  `_compute_attribute`, `trained_order.select_features`), so the numbers are identical either way
  — this is not a second implementation of the SHAP computation itself.
- **`EXPLAIN_CAP` / pairing.** Both kernel runs read the identical `corrector_targets_v3_<split>`
  file and sample with the same `seed`, so they land on the identical claim_id subset without any
  file coordinating them — see §4's own comment. Raising `EXPLAIN_CAP` (or removing the cap
  entirely by setting it above the real shared-claim count) trades runtime for using more of the
  shared population; there is nothing about the DiD math in `estimator/effect/shap_did.py` that
  requires the sample to be capped, only attribute.py/this notebook's total absence of built-in
  downsampling once ids are explicit.
- **Real row counts.** Any real claim/feature count this notebook prints comes from the company
  laptop's own real corrector_targets/log_features/processed_inputs files at run time — never
  hardcode a number seen in a local dummy run (`src/data/real/_DUMMY_DATA` marker) into this
  notebook or elsewhere; see this project's own memory on that mistake.
